# CAR171 APT editor

Alejandro S. Borlaff, NASA ARC
Adapted from: Maxime Rizzo, NASA GSFC
Date: 5/10/26

In [ ]:
import xml.etree.ElementTree as ET
import copy
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord

filename = 'CAR171.apt'
apt_file_dir = ''

CAR_dat = pd.read_csv("CAR171_apt_targets.csv")
CAR_dat
coords = SkyCoord(ra=CAR_dat['RA'].values*u.degree, dec=CAR_dat['DEC'].values*u.degree)

In [ ]:
CAR_dat

In [ ]:
coords[0].ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)

In [ ]:

import xml.etree.ElementTree as ET

def update_target_coordinates(xml_path, output_path, new_coords):
    """
    Updates RA/Dec coordinates for each FixedTarget in the XML.

    Parameters:
        xml_path (str): Path to the input XML file.
        output_path (str): Path to write the updated XML file.
        new_coords (list of tuples): List of (RA, Dec) strings.
            Example: [("13 47 3.30", "+49 01 40.49"), ...]
            Must be same length as number of FixedTarget entries.
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    # XML uses namespaces; must extract them for searching.
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all FixedTarget entries
    targets = root.findall('.//ns:FixedTarget', ns)

    if len(new_coords) != len(targets):
        raise ValueError(
            f"Provided {len(new_coords)} coordinates but XML contains {len(targets)} FixedTargets."
        )

    for (target, coord) in zip(targets, new_coords):
        ra = coord.ra.to_string(unit=u.hour, sep=' ', precision=4, pad=True)
        dec = coord.dec.to_string(unit=u.degree, sep=' ', precision=2, pad=True, alwayssign=True)
        # Find EquatorialCoordinates node
        eq = target.find('ns:EquatorialCoordinates', ns)
        if eq is not None:
            # Format must match XML's "Value" attribute: "RA Dec"
            eq.set("Value", f"{ra} {dec}")
            # print("Value", f"{ra} {dec}")
        else:
            print(f"Warning: FixedTarget missing EquatorialCoordinates element.")

    # Save modified XML
    tree.write(output_path, encoding="UTF-8", xml_declaration=True)
    print(f"Updated file written to: {output_path}")



def sync_passplan_numbers(xml_input, xml_output):
    """
    Updates each <PassPlan Number="X"> so that X matches its TargetSelection Fixed target ID.
    Example:
        <TargetSelection>Fixed: 13</TargetSelection>
        → PassPlan Number becomes "13".
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Namespace used by Roman APT XML
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Iterate through all PassPlan entries
    for passplan in root.findall('.//ns:PassPlan', ns):
        ts = passplan.find('ns:TargetSelection', ns)
        if ts is not None and "Fixed:" in ts.text:
            # Extract target number from "Fixed: N"
            target_num = ts.text.split("Fixed:")[1].strip()
            passplan.set("Number", target_num)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")



def sort_surveyplan_steps(xml_input, xml_output):
    """
    Sorts all <SurveyPlanStep> entries by their <PassPlan> value in increasing order.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Locate the <SurveyPlan> container
    survey_plan = root.find('.//ns:SurveyPlan', ns)
    if survey_plan is None:
        raise RuntimeError("Could not find <SurveyPlan> in XML.")

    # Extract all SurveyPlanStep elements
    steps = survey_plan.findall('ns:SurveyPlanStep', ns)

    # Sort by numeric PassPlan value
    def get_passplan_number(step):
        pp = step.find('ns:PassPlan', ns)
        return int(pp.text.strip()) if pp is not None else 999999999

    steps_sorted = sorted(steps, key=get_passplan_number)

    # Clear existing order
    for step in steps:
        survey_plan.remove(step)

    # Reinsert in sorted order
    for step in steps_sorted:
        survey_plan.append(step)

    # Save output
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"SurveyPlan sorted and saved to {xml_output}")



def update_orient_ranges(xml_input, xml_output, orient_min_list, orient_max_list):
    """
    Updates OrientRange OrientMin/OrientMax in each SurveyPlanStep using
    user-provided arrays.
    
    orient_min_list and orient_max_list must have the same length as the
    number of SurveyPlanStep entries.
    """

    tree = ET.parse(xml_input)
    root = tree.getroot()

    # Extract namespace automatically
    ns = {'ns': root.tag.split('}')[0].strip('{')}

    # Find all SurveyPlanStep entries
    steps = root.findall('.//ns:SurveyPlan/ns:SurveyPlanStep', ns)

    if len(steps) != len(orient_min_list) or len(steps) != len(orient_max_list):
        raise ValueError("ERROR: Input arrays must match number of SurveyPlanStep entries.")

    # Update OrientMin and OrientMax for each step
    for step, new_min, new_max in zip(steps, orient_min_list, orient_max_list):
        orient_range = step.find('ns:SpecialRequirements/ns:OrientRange', ns)
        if orient_range is not None:
            orient_range.set("OrientMin", f"{new_min} Degrees")
            orient_range.set("OrientMax", f"{new_max} Degrees")
        else:
            print(f"Warning: No OrientRange found in SurveyPlanStep uid={step.get('uid')}")

    # Save updated XML
    tree.write(xml_output, encoding="UTF-8", xml_declaration=True)
    print(f"Updated XML saved to {xml_output}")


# Example usage:
# new_min = [142.1, 143.2, 144.3, ...]
# new_max = [142.1, 143.2, 144.3, ...]


In [ ]:
sync_passplan_numbers(xml_input="CAR171.apt", xml_output="CAR171_sync.apt")
sort_surveyplan_steps(xml_input="CAR171_sync.apt", xml_output="CAR171_order.apt")
update_target_coordinates(xml_path="CAR171_order.apt", output_path="CAR171_mod.apt", new_coords=coords)
update_orient_ranges(xml_input="CAR171_mod.apt", xml_output="CAR171_orient.apt", orient_min_list=CAR_dat["V3PA_off"], orient_max_list=CAR_dat["V3PA_off"])

In [ ]:
coords

In [ ]:
def update_stray_light_APT_171(filename, apt_file_dir=apt_file_dir):
    tree = ET.parse(f'{apt_file_dir}{filename}')
    root = tree.getroot()    

    #### build target list correctly ####
    # assumes first target exists and is correctly built
    ns = {'apt': 'http://www.stsci.edu/Roman/APT'}

   # Load target file
    dat = load_targets_171()
    # n_targets = len(dat)

    for i in range(len(pp_list)): 
        print(pp_list[i].find('apt:PassPlan', ns).text)
        # observations = pp_list[i].findall('apt:Observation', ns)

        # for j in range(len(observations)):
            # observations[j].find('apt:OpticalElement', ns).text = "OPTICALELEMENTPLACE"
            # print(observations[j])


    return tree

tree = update_stray_light_APT_171(filename)

tree.write(f'{apt_file_dir}APT_1023_GhostsStrayLight_CAR171_modified.apt')

In [ ]:


#### build target list correctly ####
# assumes first target exists and is correctly built
tg = root.find('apt:Targets', ns)
targets = tg.findall('apt:FixedTarget', ns)
print(targets[0].text)

In [ ]:
pp_list[0].tag

In [ ]:
filename = "CAR171_test.apt"
tree = ET.parse(f'{apt_file_dir}{filename}')
root = tree.getroot()    
ns = {'apt': 'http://www.stsci.edu/Roman/APT'} 
pp = root.find('apt:PassPlans', ns)
pp_list = pp.findall('apt:PassPlan', ns) 
dat = load_targets_171()
n_targets = len(dat)

for i in range(len(pp_list)): 
    # print(i)
    observations = pp_list[i].findall('apt:Observation', ns)
    for j in range(len(observations)):
        # observations[j].find('apt:OpticalElement', ns).attrib['OpticalElement'] = "OPTICALELEMENTPLACE"
        observations[j].find('apt:OpticalElement', ns).text = "OPTICALELEMENTPLACE"
        #observation_fields = observations[j].findall('apt:OpticalElement', ns)
        #print(observations[j].find('apt:OpticalElement', ns).text)
        #observation_fields[0].set('apt:OpticalElement', "BLAHBLAH")

tree.write(f'{apt_file_dir}TEST_APT.apt')

    #pp_target = copy.deepcopy(pp_list[0])
    #observation_fields = pp.findall('apt:OpticalElement', ns) 
    # print(pp_target.find('apt:TargetSelection', ns).text)
    # print(pp_target.find('apt:Observation', ns).text)
    #print(observation_fields)
# obs_list = root.findall('apt:Observation', ns)
# print(obs_list)

In [ ]:
apt_file_dir = ''
tree = ET.parse(f'{apt_file_dir}CAR171.apt')
root = tree.getroot()

In [ ]:
print(ET.tostring(root))

In [ ]:
def back_update_stray_light_APT_171(filename, apt_file_dir=apt_file_dir):
    tree = ET.parse(f'{apt_file_dir}{filename}')
    root = tree.getroot()    

    #### build target list correctly ####
    # assumes first target exists and is correctly built
    ns = {'apt': 'http://www.stsci.edu/Roman/APT'}
    tg = root.find('apt:Targets', ns)

   # Load target file
    dat = load_targets_171()
    n_targets = len(dat)

    #### Now build pass plans correctly ####
    pp = root.find('apt:PassPlans', ns)
    pp_list = pp.findall('apt:PassPlan', ns)
    pp_target = copy.deepcopy(pp_list[0])

    for p in pp_list:
        pp.remove(p)
    

    # now create the pass plans back, pointing at correct targets and with correct names
    for i in range(n_targets):
        first = copy.deepcopy(pp_target)
        first.attrib['Number'] = str(i+1)
        first.find('apt:Label', ns).text = f'Target_{i+1}'
        first.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'
        pp.append(first)
 

    # Add the resultants to all Pass Plans
    # If CAR.171.1 = MA Table = IM_171_10 = Resultant 10 
    # If CAR.171.2 = MA Table = IM_193_11 = Resultant 11

    #### Now create the survey ####
    sp = root.find('apt:SurveyPlan', ns)
    steps = sp.findall('apt:SurveyPlanStep', ns)
    step_model = copy.deepcopy(steps[0])

    # first clean up
    for step in steps:
        sp.remove(step)
    
    # observe the target
    for i in range(n_targets):
        first_step = copy.deepcopy(step_model)
        first_step.find('apt:PassPlan', ns).text = str(i+1)
        s = first_step.find('apt:SpecialRequirements', ns)
        s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['V3PA_off']:.4f} Degrees"
        s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['V3PA_off']:.4f} Degrees"        
        sp.append(first_step)

    num_steps = 0
    for sp in root.iterfind('apt:SurveyPlan', ns):
        for step in sp.iterfind('apt:SurveyPlanStep', ns):
            num_steps+=1
    print(f'Number of survey steps: {num_steps}')




    return tree

tree = update_stray_light_APT_171(filename)

tree.write(f'{apt_file_dir}APT_1023_GhostsStrayLight_CAR171_modified.apt')

In [ ]:
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}
for sp in root.iterfind('apt:SurveyPlan', ns):
    print(sp)

In [ ]:
### Scratch - for practice only ###
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}
for sp in root.iterfind('apt:SurveyPlan', ns):
        for links in sp.iterfind('apt:Links', ns):
            for lr in links.iterfind('apt:LinkReq', ns):
                # print(lr.attrib.items(),lr.attrib['Type'], lr.attrib['Min'], lr.attrib['Max'])
                #print(lr.attrib['Type'], lr.attrib['Min'], lr.attrib['Max'])
                for ss in lr.iterfind('apt:SurveyStep', ns):
                    role = ss.get('Role')
                    step_number = ss.text
                    print(role, step_number)

test = copy.deepcopy(lr)
test
for ss in test.iterfind('apt:SurveyStep', ns):
    role = ss.get('Role')
    if role=='Reference':
        ss.text = 20
    elif role=='Oriented':
        ss.text = 21


for ss in test.iterfind('apt:SurveyStep', ns):
    role = ss.get('Role')
    step_number = ss.text
    print(role, step_number)
print(lr.attrib.items())


i=0
for sp in root.iterfind('apt:SurveyPlan', ns):
        for links in sp.iterfind('apt:Links', ns):
            for lr in links.iterfind('apt:LinkReq', ns):
                 i+=1
                 print(i)

sp = root.find('apt:SurveyPlan', ns)
print(sp)
links = sp.find('apt:Links', ns)
print(links)
linkreq_list = links.findall('apt:LinkReq', ns)
print(linkreq_list)
tg = root.find('apt:Targets', ns)
targets = tg.findall('apt:FixedTarget', ns)
print(len(targets))
# for tgt in targets:
#      tg.remove(tgt)

# dat = load_targets(filename='RST_instrument_model/Pitch-raster.csv')
# for i in range(len(dat)):
#      new_tg = copy.deepcopy(target_model)
#      new_tg.find('apt:Number', ns).text = i+1
#      new_tg.find('apt:TargetName', ns).text = dat.iloc[i]['Name']
#      new_tg.find('apt:Comments', ns).text = dat.iloc[i]['Comments']
#      new_tg.find('apt:Category', ns).text = dat.iloc[i]['Category']
#      new_tg.find('apt:Keywords', ns).text = dat.iloc[i]['Description']
#      new_tg.find('apt:EquatorialCoordinates', ns).text = f"{dat.iloc[i]['RA']} {dat.iloc[i]['DEC']}"
#      tg.append(new_tg)
     
pp = root.find('apt:PassPlans', ns)
pp_list = pp.findall('apt:PassPlan', ns)
pp_target_only = copy.deepcopy(pp_list[0])
pp_prism_first = copy.deepcopy(pp_list[1])
pp_prism_last = copy.deepcopy(pp_list[2])

# list_fixed_target_attributes(target_model)

In [ ]:
sp = root.find('apt:SurveyPlan', ns)
print(sp)
for step in sp.iterfind('apt:SurveyPlanStep', ns):
    print(step)

In [ ]:
# dat = pd.read_csv('Pitch-raster2.csv',  header=None, index_col=False, names=['Name', 'Category', 'Description', 'RA', 'DEC', 'Comments', 'Pitch', 'Roll'])
# dat

In [ ]:
dat.iloc[10]['DEC']

In [ ]:
filename = 'APT_1022_SolarStrayLight_TSR.apt'
apt_file_dir = ''
ns = {'apt': 'http://www.stsci.edu/Roman/APT'}

target_file = 'Pitch-raster5_v2.csv'

def update_stray_light_APT(filename, rolls=[-14.9, -7.4, 0.0, 7.4, 14.9], apt_file_dir=apt_file_dir, orient_range=0.0):
    tree = ET.parse(f'{apt_file_dir}{filename}')
    root = tree.getroot()    
    n_orient= len(rolls)

    #### build target list correctly ####
    # assumes first target exists and is correctly built
    tg = root.find('apt:Targets', ns)
    targets = tg.findall('apt:FixedTarget', ns)

    # grab first target as model
    # make sure this one has the boresight correctly set
    target_model = copy.deepcopy(targets[0])
    # delete all targets
    for tgt in targets:
        tg.remove(tgt)

    # Load target file
    dat = load_targets(filename=target_file)
    n_targets = len(dat)

    # add targets one by one
    for i in range(len(dat)):
        new_tg = copy.deepcopy(target_model)
        new_tg.find('apt:Number', ns).text = str(i+1)
        new_tg.find('apt:TargetName', ns).text = dat.iloc[i]['Name']
        new_tg.find('apt:Comments', ns).text = dat.iloc[i]['Comments']
        new_tg.find('apt:Category', ns).text = dat.iloc[i]['Category']
        new_tg.find('apt:Keywords', ns).text = dat.iloc[i]['Description']
        new_tg.find('apt:EquatorialCoordinates', ns).attrib['Value'] = f"{dat.iloc[i]['RAstr']} {dat.iloc[i]['DECstr']}"
        tg.append(new_tg)

    #### Now build pass plans correctly ####
    # Assumes first 3 pass plans exist and are correctly built: one with just the target, one with the prism first, and one with the prism last
    pp = root.find('apt:PassPlans', ns)
    pp_list = pp.findall('apt:PassPlan', ns)
    pp_target_only = copy.deepcopy(pp_list[0])
    pp_prism_first = copy.deepcopy(pp_list[1])
    pp_prism_last = copy.deepcopy(pp_list[2])
    for p in pp_list:
        pp.remove(p)
    
    # now create the pass plans back, pointing at correct targets and with correct names
    for i in range(n_targets):
        first = copy.deepcopy(pp_target_only)
        second = copy.deepcopy(pp_prism_first)
        third = copy.deepcopy(pp_prism_last)

        first.attrib['Number'] = str(i*3+1)
        second.attrib['Number'] = str(i*3+2)
        third.attrib['Number'] = str(i*3+3)

        first.find('apt:Label', ns).text = f'Target_{i+1}'
        second.find('apt:Label', ns).text = f'Target_{i+1}_wPrismFirst'
        third.find('apt:Label', ns).text = f'Target_{i+1}_wPrismLast'
        first.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'
        second.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'
        third.find('apt:TargetSelection', ns).text = f'Fixed: {i+1}'

        pp.append(first)
        pp.append(second)
        pp.append(third)

    #### Now create the survey ####
    sp = root.find('apt:SurveyPlan', ns)
    steps = sp.findall('apt:SurveyPlanStep', ns)
    step_model = copy.deepcopy(steps[0])
    for step in steps:
        sp.remove(step)
    
    # for each orient in roll, observe the target
    for i in range(n_targets):

        roll_list = copy.deepcopy(rolls)

        if i%2 == 1:
            roll_list = roll_list[::-1]

        first_step = copy.deepcopy(step_model)
        first_step.find('apt:PassPlan', ns).text = str(i*3+2)
        s = first_step.find('apt:SpecialRequirements', ns)
        if dat.iloc[i]['DEC']<0: # flip the signs for continuity of schedulability
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[0]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[0]+orient_range:.4f} Degrees"
        else:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[0]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[0]+orient_range:.4f} Degrees"
        
        sp.append(first_step)

        for j in range(n_orient-2):
            new_step = copy.deepcopy(step_model)
            new_step.find('apt:PassPlan', ns).text = str(i*3+1)
            s = new_step.find('apt:SpecialRequirements', ns)
            if dat.iloc[i]['DEC']<0:
                s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[1+j]-orient_range:.4f} Degrees"
                s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[1+j]+orient_range:.4f} Degrees"
            else:
                s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[1+j]-orient_range:.4f} Degrees"
                s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[1+j]+orient_range:.4f} Degrees"
            sp.append(new_step)
        
        last_step = copy.deepcopy(step_model)
        last_step.find('apt:PassPlan', ns).text = str(i*3+3)
        s = last_step.find('apt:SpecialRequirements', ns)
        if dat.iloc[i]['DEC']<0:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']+roll_list[-1]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']+roll_list[-1]+orient_range:.4f} Degrees"
        else:
            s.find('apt:OrientRange', ns).attrib['OrientMin'] = f"{dat.iloc[i]['Roll']-roll_list[-1]-orient_range:.4f} Degrees"
            s.find('apt:OrientRange', ns).attrib['OrientMax'] = f"{dat.iloc[i]['Roll']-roll_list[-1]+orient_range:.4f} Degrees"
        sp.append(last_step)

    num_steps = 0
    for sp in root.iterfind('apt:SurveyPlan', ns):
        for step in sp.iterfind('apt:SurveyPlanStep', ns):
            num_steps+=1
    print(f'Number of survey steps: {num_steps}')

    links = sp.find('apt:Links', ns)

    # copy the first linking requirement as our model
    # lr_list = links.findall('apt:LinkReq', ns)
    # lr_model = copy.deepcopy(lr_list[0])

    # clean up all Linking requirements to start fresh
    for item in links.findall('apt:LinkReq', ns):
        links.remove(item)

    # now create sets of linking requirements between each n_orient+1 targets
    ### OBSOLETE - an earlier iteration was doing this but we are switching to absolute v3pa instead
    # for nsteps in range(num_steps//(n_orient+1)):
    #     for orient in range(n_orient):
    #         added_req = copy.deepcopy(lr_model)
    #         added_req.attrib['Min'] = f"{orient_list[orient][0]} Degrees"
    #         added_req.attrib['Max'] = f"{orient_list[orient][1]} Degrees"
    #         for ss in added_req.iterfind('apt:SurveyStep', ns):
    #             role = ss.get('Role')
    #             if role=='Reference':
    #                 ss.text = str(nsteps*(n_orient+1)+orient+1)
    #             elif role=='Oriented':
    #                 ss.text = str(nsteps*(n_orient+1)+orient+2)
            
    #         links.append(added_req)


    return tree
    

tree = update_stray_light_APT(filename, rolls=[-14.5, -7.4, 0.0, 7.4, 14.5], apt_file_dir=apt_file_dir)
    
tree.write(f'{apt_file_dir}APT_118_SolarStrayLight_TSR_final_v2.apt')